In [ ]:
# === 1. Standalone Environment Setup ===
import sys, os
from pathlib import Path

try:
    # Colab Environment
    import google.colab
    from google.colab import drive, runtime
    drive.mount('/content/drive')
    
    # Clone repository if it doesn't exist
    if not os.path.exists('/content/deep-learning'):
        !git clone --depth 1 -q https://github.com/khangnghiem/deep-learning.git /content/deep-learning
    
    REPO_ROOT = Path('/content/deep-learning')
    os.chdir(REPO_ROOT / 'experiments/016_polyp_fast_diag')
    
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))
        
except ImportError:
    # Local Environment
    cur = Path().resolve()
    REPO_ROOT = cur.parent if cur.name in ('explorations', 'experiments') else cur.parents[1]
    if str(REPO_ROOT) not in sys.path:
        sys.path.insert(0, str(REPO_ROOT))

from src.config.paths import GOLD, BRONZE, SILVER, DATA_LAKE, MODELS, PRETRAINED, TRAINED, OPS, MLFLOW_TRACKING_URI, REPOS


In [ ]:
# Cell 2: Install dependencies
!pip install -q ultralytics mlflow


In [ ]:
# Cell 3: Check Gold Data
import os
data_path = str(GOLD / '016_polyp_fast_diag_dataset')
if not os.path.exists(data_path):
    print("Gold dataset not found!")
!cat {data_path}/dataset.yaml


In [ ]:
# Cell 4: Hyperparameter Tuning (Optional)
# Uncomment the lines below to run hyperparameter tuning before training.
# Tuning will take a significant amount of time (e.g., 30+ iterations).
# !python train.py \
#     --dataset /content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset \
#     --output /content/drive/MyDrive/models/trained/016_polyp_fast_diag_tune \
#     --model yolo11m-seg.pt \
#     --tune \
#     --epochs 30 \
#     --iterations 30 \
#     --batch-size 32 \
#     --img-size 640


In [ ]:
# Cell 5: Train YOLO Model
import mlflow
mlflow.set_tracking_uri('file:///content/drive/MyDrive/mlflow/mlruns')
mlflow.set_experiment('016_polyp_fast_diag')
with mlflow.start_run(run_name="yolo11m-seg_run_epochs200"):
    !MLFLOW_TRACKING_URI=file:///content/drive/MyDrive/mlflow/mlruns MLFLOW_EXPERIMENT_NAME=016_polyp_fast_diag python train.py \
        --dataset /content/drive/MyDrive/data_lake/03_gold/016_polyp_fast_diag_dataset \
        --output /content/drive/MyDrive/models/trained/016_polyp_fast_diag \
        --model yolo11m-seg.pt \
        --epochs 200 \
        --batch-size 32 \
        --img-size 640


In [ ]:
# Cell 6: Release resources
from google.colab import runtime
runtime.unassign()
